# RAPTOR Chunking Arms — Experiment Runner (Colab)

Compares 5 leaf-chunking arms inside an otherwise **frozen** RAPTOR pipeline, isolating the
chunker as the only variable. Same gpt-oss model (via OpenRouter) is used for summarization
**and** QA across all arms, so results are directly comparable.

| Arm | Chunker |
|---|---|
| `token` | original RAPTOR sentence/token splitter (**baseline RAPTOR**) |
| `structure` | always-LLM structure chunking (char-offset map + per-section repair) |
| `ahc` | **Adaptive Hybrid Chunking** (structure score → route → repair) — *our architecture* |
| `semantic` | SBERT embedding-boundary chunking |
| `flat` | token chunks + FAISS, no tree (contextualizes tree value) |

Datasets & metrics (faithful to arXiv:2401.18059v1): **QASPER** → Answer-F1; **QuALITY** →
accuracy (+ HARD); **NarrativeQA** → ROUGE-L / BLEU-1/4 / METEOR. Retrieval = collapsed tree,
2000-token budget.

> **Tip:** the LLM runs over the API, so a GPU runtime is optional (only SBERT embeddings use it).
> Run the small **smoke test** first to validate the whole pipeline. Every LLM call is cached on
> disk, so reruns are cheap and the run is resumable.

## 1. Clone the repo and install the full stack

In [ ]:
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

## 2. Credentials + METEOR data

We use **Cerebras** as the inference provider — it serves the **same `gpt-oss-120b`** model on a genuinely free tier (**1,000,000 tokens/day, no credit card**), which is ~20× more usable than OpenRouter's free tier (50 requests/day without credits). Same open-weights model → results stay comparable.

Get a free key at **https://cloud.cerebras.ai** → *API Keys*. (Groq — https://console.groq.com — also serves `gpt-oss-120b` free and is a drop-in fallback, but its 200K tokens/day is ~5× smaller.)

In [ ]:
import os, getpass
# FREE, no credit card: https://cloud.cerebras.ai -> API Keys.
# Cerebras serves gpt-oss-120b free at 1M tokens/day (vs OpenRouter's 50 req/day).
os.environ['CEREBRAS_API_KEY'] = getpass.getpass('Cerebras API key: ')

import nltk
for pkg in ('wordnet', 'omw-1.4', 'punkt'):
    nltk.download(pkg, quiet=True)
print('ready')

## 2b. Persist cache + results to Google Drive (recommended)

Free-tier Colab sessions recycle, and OpenRouter free-tier rate limits mean a full run spans many sittings. Mounting Drive keeps `.llm_cache/` and `results/` across disconnects, so every rerun **resumes from cache** instead of re-spending LLM calls. Authorize Drive access when prompted. (Off Colab this falls back to a local `./raptor_runs`.)

In [ ]:
# Persist the LLM cache + results to Google Drive so a disconnect never loses
# the (expensive) cached calls. Re-running then resumes from cache.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/raptor_runs'
except Exception as e:
    print('Not on Colab / Drive unavailable -> using local ./raptor_runs', e)
    RUN_DIR = 'raptor_runs'

import os
CACHE_DIR = os.path.join(RUN_DIR, '.llm_cache')
RESULTS_DIR = os.path.join(RUN_DIR, 'results')
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('RUN_DIR   =', RUN_DIR)
print('CACHE_DIR =', CACHE_DIR)

## 3. (Optional) Smoke-test the harness offline
All logic is unit-tested without network/model loads; run it to confirm the clone is intact.

In [ ]:
!python -m pytest tests/ -q

## 4. Configure — start with the fast smoke test

This first config is deliberately tiny: **`token` (baseline RAPTOR) vs `ahc` (our method)** on **QuALITY only**, **2 docs**, 1 seed. QuALITY is short multiple-choice, so this exercises the whole API path end-to-end (build tree → summarize → retrieve → answer → score) in minutes — *before* you commit hours to the full grid. Scale up in section 8 once this is green.

Caching is keyed by `(model, prompt)`, so nothing here is wasted: the calls this run makes are reused by the bigger runs.

In [ ]:
from experiments.config import ExperimentConfig

# SMOKE TEST (free Cerebras tier) — token (baseline) vs ahc (ours), QuALITY, 1 doc.
# Purpose: confirm your Cerebras key + model id work and the live path runs end-to-end
# (build tree -> summarize -> retrieve -> answer -> score). Auth/model errors surface on
# the FIRST call (seconds); a full pass is ~10 min at 5 req/min. Every call is cached.
cfg = ExperimentConfig(
    model='gpt-oss-120b',                       # Cerebras model id
    base_url='https://api.cerebras.ai/v1',      # OpenAI-compatible endpoint
    api_key_env='CEREBRAS_API_KEY',
    arms=['token', 'ahc'],
    datasets=['quality'],
    subset_sizes={'quality': 1},
    seeds=[0],
    retrieval_max_tokens=2000,                  # paper's collapsed-tree main setting
    leaf_max_tokens=100,
    tau=0.5,                                    # AHC routing threshold
    cache_dir=CACHE_DIR,                        # on Drive (section 2b)
    results_dir=RESULTS_DIR,
)
cfg.to_dict()

## 4b. Verify dataset alignment with the paper (run once)

Confirms QuALITY is scored the paper's way — **accuracy over the dev split, plus the QuALITY-HARD subset** — and that gold labels + the HARD flag actually populated from the HF mirror. A silent field-name mismatch would otherwise make accuracy read **0**; the `assert` below fails loudly instead. (QuALITY-HARD = questions a majority of annotators got wrong under time pressure — the dataset's `difficult` flag.)

In [ ]:
from experiments.datasets import get_loader

ql = get_loader('quality').load(limit=5)          # dev/validation split
qs = [q for d in ql for q in d.questions]
with_gold = sum(q.gold_index is not None for q in qs)
hard = sum(q.is_hard for q in qs)
print(f'QuALITY dev: {len(ql)} docs, {len(qs)} questions')
print(f'  with gold label : {with_gold}/{len(qs)}   (must be > 0)')
print(f'  HARD (difficult): {hard}/{len(qs)}')
print(f'  options/question: {sorted({len(q.options or []) for q in qs})}   -> chance acc = 0.25')
assert with_gold > 0, 'No gold labels parsed -> accuracy would be a silent 0. Check the mirror schema.'
print('alignment OK: accuracy + QuALITY-HARD will compute correctly.')

## 4c. Recommended free-tier ~1-hour pilot — `token` vs `ahc` on QuALITY (Cerebras)

The full 5-arm × 3-dataset grid is thousands of calls and won't finish on any free tier in an hour. This is the highest-value result that *fits* a free, ~1-hour budget: **`token` (baseline RAPTOR)** vs **`ahc` (our method)** on **QuALITY** — the cheapest, cleanest comparison (short passages, many MCQs each, one clear accuracy metric). NarrativeQA is excluded (full books → hundreds of summary calls each, and they'd blow past the free token cap).

**You get:** overall accuracy + QuALITY-HARD accuracy + AHC structure-routing rate, with bootstrap CIs, on ~100 questions/arm — a real answer to *"does AHC move accuracy vs baseline RAPTOR on the same questions and reader?"* (Pilot-scale: CIs are wide; it's a signal, not a final number.)

> **Why Cerebras (free, no card).** It serves the **same `gpt-oss-120b`** at **1M tokens/day** — vs OpenRouter's 50 requests/day without credits, which can't finish even the 2-doc smoke. Limits are **5 req/min · 30K tok/min · 1M tok/day**. The 5 req/min cap is what bounds a sitting to ~6 docs/hour; the 1M tok/day resets daily.

> **Free-tier is slow on purpose — lean on resume.** At 5 req/min you'll see frequent `429`s; the harness backs off and self-paces automatically (and a stray content block is recorded + skipped, never fatal). Because the cache + results live on Drive and `resume=True` skips completed docs, you can **run ~1 hr/day for a few days** and accumulate a much stronger sample with zero rework.

**How to run:** §3 pytest + §4 (2-doc smoke) for a few-minute sanity check → then **§4c → §5 → §6** for the ~1-hour result.

In [ ]:
# ============================================================
# RECOMMENDED FREE-TIER ~1-HOUR PILOT (Cerebras) — a real, comparable result.
#   token (baseline RAPTOR)  vs  ahc (our method)  on QuALITY.
#   Cerebras free = 5 req/min, 1M tokens/day -> ~300 calls / hour -> ~6 docs.
#   leaf_max_tokens stays 100 (paper-faithful) so numbers stay comparable.
# ============================================================
from experiments.config import ExperimentConfig

cfg = ExperimentConfig(
    model='gpt-oss-120b',                       # SAME open-weights model -> comparable
    base_url='https://api.cerebras.ai/v1',
    api_key_env='CEREBRAS_API_KEY',
    arms=['token', 'ahc'],                      # baseline RAPTOR vs our contribution
    datasets=['quality'],                       # cheapest + cleanest metric (accuracy)
    subset_sizes={'quality': 6},                # ~1 hr at 5 req/min. See note below.
    seeds=[0],
    retrieval_max_tokens=2000,                  # paper collapsed-tree setting
    leaf_max_tokens=100,                        # paper-faithful -- do NOT raise
    tau=0.5,                                    # AHC routing threshold
    cache_dir=CACHE_DIR,
    results_dir=RESULTS_DIR,
)
# Sizing: Cerebras free caps throughput at 5 req/min, so a single sitting fits ~6 docs.
#   * If section 5 finishes with time left, just raise 'quality' (e.g. to 9) and re-run
#     5 -> 6: resume adds ONLY the new docs (already-done ones are skipped).
#   * Across days: 1M tokens/day resets daily and the run is checkpointed on Drive, so
#     run ~1 hr/day for 3 days -> ~18 docs (~300+ questions/arm) for a stronger result.
cfg.to_dict()

## 5. Run
Builds one tree per (arm, dataset, doc) — SBERT embeddings + gpt-oss summaries — then answers
every question with the same gpt-oss reader and scores it.

**Robust by construction** — a long run is no longer a fragile black box:
- **Progress:** a `tqdm` bar tracks each `dataset/arm`, plus a per-dataset/arm line.
- **Crash-safe + resumable:** results are checkpointed to the results file after *every doc*, and `resume=True` skips already-completed `(dataset, arm, doc)` on a rerun — so a Colab disconnect costs you at most one doc.
- **Moderation-proof:** the `:free` model is served by a provider that force-moderates input. If a question is rejected (HTTP 403, *"flagged for …"*), it is **not** retried 6× and it does **not** kill the grid — it is recorded as `blocked` and the run continues. (Previously this single 403 crashed the entire run after an hour.)
- **Permanent vs transient errors:** 403/4xx fail fast; only 429/5xx and transient errors back off and retry.

In [ ]:
from experiments import runner, report

# resume=True       -> reconnect-safe: completed (dataset, arm, doc) are skipped and
#                      the results file is checkpointed after every doc.
# show_progress=True -> tqdm bar per dataset/arm so a long run isn't a black box.
results = runner.run(cfg, seed=cfg.seeds[0], resume=True, show_progress=True)

recs = results['records']
n_blocked = sum(1 for r in recs if r.get('blocked'))
n_err = sum(1 for r in recs if r.get('error') and not r.get('blocked'))
print(f"records: {len(recs)}  |  moderation-blocked: {n_blocked}  |  other errors: {n_err}")

## 6. Results: per-arm tables + AHC routing rate

**Fairness note — uniform drop of blocked questions.** Moderation runs on the *retrieved context*, which differs per arm, so a question can be blocked under one arm yet answered under another. Scoring it only where it survived would bias the comparison. So `aggregate` drops any question blocked (or errored) under **any** arm from **all** arms — every arm is scored on the identical question set. `report.blocked_report(...)` tells you how many questions were dropped per dataset.

In [ ]:
agg = report.aggregate(results['records'])            # uniform drop applied by default
routing = report.routing_rate(results['records'])
dropped = report.blocked_report(results['records'])   # per-dataset moderation-drop counts
print(report.to_markdown(agg, routing, dropped))
if dropped:
    print('\ndropped (blocked under >=1 arm, removed from all):', dropped)

### Reading your numbers against the RAPTOR paper

Your reader is `gpt-oss-120b`, **not** the paper's GPT-4 / GPT-3 / UnifiedQA, so *absolute* numbers will not match a leaderboard — this is a **controlled arm-vs-arm** study (like the paper's own Tables 2 & 4), where only the chunker changes. Use the paper as a sanity band, not a target. The question that matters: does `ahc` move the metric vs `token` on the **same split, same reader**?

**QuALITY — accuracy (chance = 25%).** This harness reports the **dev** split (reproducible; the test set needs a leaderboard submission).

| Setting (source) | Acc |
|---|---:|
| BM25 + UnifiedQA — dev (Table 4) | 49.9 |
| DPR + UnifiedQA — dev (Table 4) | 53.9 |
| RAPTOR + UnifiedQA-3B — dev (Table 4) | 56.6 |
| RAPTOR + GPT-3 — dev (Table 4) | 62.4 |
| RAPTOR + GPT-4 — **test** set (Table 7) | 82.6 |
| RAPTOR + GPT-4 — **test**, QuALITY-HARD (Table 7) | 76.2 |

> The 82.6 / 76.2 headline is GPT-4 on the **hidden test set** — do **not** compare your dev number to it directly. The comparable rows are the dev ones (56.6 / 62.4).

**QASPER** — Answer token-F1 (RAPTOR): UnifiedQA 36.6 · GPT-3 53.1 · GPT-4 55.7 (Tables 3 & 5).
**NarrativeQA** — RAPTOR + UnifiedQA-3B (Table 6): ROUGE-L 30.8 · BLEU-1 23.5 · BLEU-4 6.4 · METEOR 19.1.

## 7. Resuming after a disconnect

Recovery is automatic on two levels. **(1) LLM cache:** because `.llm_cache/` lives on Drive (section 2b), every already-completed call is served from disk on rerun — only missing calls hit the network. **(2) Run checkpoint:** the results file is rewritten after *every doc*, and `runner.run(..., resume=True)` (the default) skips any `(dataset, arm, doc)` already present in it. So after a reconnect, just re-run sections **1, 2, 2b, 4, 5, 6** — completed docs are skipped, not recomputed. Stop and resume as many times as the free tier forces you to.

In [ ]:
# Nothing to copy if you mounted Drive in section 2b — cache + results persist there.
# Sanity-check what has accumulated so far:
import os
print('cached LLM calls:', len(os.listdir(CACHE_DIR)) if os.path.isdir(CACHE_DIR) else 0)
print('results files:   ', os.listdir(RESULTS_DIR) if os.path.isdir(RESULTS_DIR) else [])

## 8. Scale up to the full study

Re-run sections **4 → 5 → 6** with a bigger config. Keep `cache_dir=CACHE_DIR` / `results_dir=RESULTS_DIR` so results accumulate on Drive. On the free tier, run **one arm or one dataset at a time** — the cache makes this seamless, and `resume=True` (default) means an interrupted grid picks up where it left off.

**Pilot (all 5 arms, 3 docs/dataset) — validates every arm + metric:**
```python
cfg = ExperimentConfig(
    model='openai/gpt-oss-120b:free',
    arms=['token', 'structure', 'ahc', 'semantic', 'flat'],
    datasets=['qasper', 'quality', 'narrativeqa'],
    subset_sizes={'qasper': 3, 'quality': 3, 'narrativeqa': 3},
    seeds=[0],
    cache_dir=CACHE_DIR, results_dir=RESULTS_DIR,
)
results = runner.run(cfg, seed=0)                      # resume + tqdm on by default
print(report.to_markdown(report.aggregate(results['records']),
                         report.routing_rate(results['records']),
                         report.blocked_report(results['records'])))
```

**Full study (paper subsets, 3 seeds → mean±std + bootstrap CIs):**
```python
cfg = ExperimentConfig(
    model='openai/gpt-oss-120b',   # paid; the free tier will rate-limit a run this size
    subset_sizes={'qasper': 50, 'quality': 50, 'narrativeqa': 25},
    seeds=[0, 1, 2],
    cache_dir=CACHE_DIR, results_dir=RESULTS_DIR,
)
for s in cfg.seeds:
    runner.run(cfg, seed=s)
```

> **Reality check:** the full 5-arm × 3-dataset × 3-seed grid is thousands of LLM calls. The `:free` model *will* rate-limit it into many sittings (the cache + `resume=True` make that survivable). For a run you want finished in one or two sessions, swap to the paid `openai/gpt-oss-120b` — one line, same results. The paid route is also **not** force-moderated, so you won't see `blocked` questions there.